# Image Representation, PCA, Eigenfaces and Face Recognition
Dataset: AT&T (ORL) Database of Faces, 40 subjects x 10 images, 112 x 92 grayscale.


In [1]:
import os, sys, json, glob
import numpy as np
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = os.environ.get("ORL_DIR", "data/att_faces")
FIG_DIR, RES_DIR = "figures", "results"
SEED = 42
N_TRAIN_PER_SUBJECT = 8          # 8 train / 2 test per subject
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
summary = {}

## Part A: Image representation

In [2]:
def load_orl(root):
    subjects = sorted(glob.glob(os.path.join(root, "s*")),
                      key=lambda p: int(os.path.basename(p)[1:]))
    if not subjects:
        raise FileNotFoundError(
            f"No s1..s40 folders found in '{root}'. Download the AT&T/ORL faces "
            "and place them there (or set ORL_DIR).")
    imgs, labels = [], []
    for p in subjects:
        sid = int(os.path.basename(p)[1:])
        files = sorted(glob.glob(os.path.join(p, "*.pgm")) + glob.glob(os.path.join(p, "*.png")),
                       key=lambda f: int(os.path.splitext(os.path.basename(f))[0]))
        for f in files:
            imgs.append(np.array(Image.open(f).convert("L"), dtype=np.float64))
            labels.append(sid)
    return np.stack(imgs), np.array(labels)

images, y = load_orl("D:\\202618006_AvdhiShah_DS605\\202618006_lab_07\\data")
n, H, W = images.shape
D = H * W
print(f"Images: {images.shape}  (n={n}, height={H}, width={W}, pixels={D})")
print(f"Intensity range: min={images.min():.0f} max={images.max():.0f} dtype=uint8-derived")
print(f"Subjects: {len(np.unique(y))}, images per subject: {np.bincount(y)[1:].min()}-{np.bincount(y)[1:].max()}")
print("Top-left 5x5 pixel block of image 0:\n", images[0, :5, :5].astype(int))

X = images.reshape(n, D) / 255.0          # flattened feature vectors in [0, 1]
print("Image matrix X:", X.shape, "| labels:", y.shape)
summary.update(n_images=int(n), height=int(H), width=int(W), n_pixels=int(D),
               n_subjects=int(len(np.unique(y))))

# Reproducible per-subject split: every subject is in both sets
train_idx, test_idx = [], []
for s in np.unique(y):
    idx = rng.permutation(np.where(y == s)[0])
    train_idx += list(idx[:N_TRAIN_PER_SUBJECT])
    test_idx += list(idx[N_TRAIN_PER_SUBJECT:])
train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)
Xtr, ytr, Xte, yte = X[train_idx], y[train_idx], X[test_idx], y[test_idx]
assert set(ytr) == set(yte) == set(y)
print(f"Train: {Xtr.shape}, Test: {Xte.shape}; subjects in both sets: {len(set(ytr) & set(yte))}")
summary.update(n_train=int(len(ytr)), n_test=int(len(yte)), seed=SEED)

fig, ax = plt.subplots(2, 5, figsize=(10, 4.5))
for a, i in zip(ax.ravel(), range(10)):
    a.imshow(images[i], cmap="gray"); a.axis("off"); a.set_title(f"s{y[i]} #{i+1}", fontsize=8)
plt.suptitle("Sample images of subject 1"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/00_sample_faces.png", dpi=130); plt.close()

Images: (400, 112, 92)  (n=400, height=112, width=92, pixels=10304)
Intensity range: min=0 max=251 dtype=uint8-derived
Subjects: 40, images per subject: 10-10
Top-left 5x5 pixel block of image 0:
 [[48 49 45 47 49]
 [45 52 39 46 56]
 [45 50 42 51 51]
 [49 46 47 47 50]
 [46 46 47 48 48]]
Image matrix X: (400, 10304) | labels: (400,)
Train: (320, 10304), Test: (80, 10304); subjects in both sets: 40


## Part B: PCA, Eigenffaces and Compression
### Mean face and centering

In [3]:
mean_face = Xtr.mean(axis=0)
Xc = Xtr - mean_face
m = Xc.shape[0]
plt.figure(figsize=(3, 3.6)); plt.imshow(mean_face.reshape(H, W), cmap="gray")
plt.title("Mean face"); plt.axis("off"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_mean_face.png", dpi=130); plt.close()

### using covariance / eigendecomposition
The full covariance is D x D (10304 x 10304), but its rank is at most m-1.
We eigendecompose the small m x m Gram matrix `Xc Xc^T/(m-1)` (Turk-Pentland trick).
If `Xc Xc^T u = l u`, then `v = Xc^T u` is an eigenvector of `Xc^T Xc` with the same eigenvalue.

In [4]:
G = (Xc @ Xc.T) / (m - 1)
evals, U = np.linalg.eigh(G)
order = np.argsort(evals)[::-1]
evals, U = evals[order], U[:, order]
evals = np.clip(evals, 0, None)
keep = evals > 1e-10 * evals[0]            # drop numerically-zero directions (rank <= m-1)
evals, U = evals[keep], U[:, keep]
V_eig = Xc.T @ U
V_eig /= np.linalg.norm(V_eig, axis=0, keepdims=True)    # (D, r) unit eigenfaces
print("Non-zero components (rank):", len(evals))

# Sanity check on a subset of the true covariance: C v = lambda v
C_v = (Xc.T @ (Xc @ V_eig[:, :5])) / (m - 1)
print("Covariance eigen-equation residual (top 5):",
      np.linalg.norm(C_v - V_eig[:, :5] * evals[:5], axis=0))

Non-zero components (rank): 319
Covariance eigen-equation residual (top 5): [2.11103524e-14 3.32313547e-14 1.73429871e-14 1.09056721e-14
 1.40057743e-14]


### using SVD of the centered data matrix

In [5]:
U_s, S, Vt = np.linalg.svd(Xc, full_matrices=False)
evals_svd = S**2 / (m - 1)
V_svd = Vt.T
r = len(evals)

### Comparison of the two approaches

In [6]:
r_cmp = min(r, len(S))
eig_err = np.max(np.abs(evals[:r_cmp] - evals_svd[:r_cmp]) / evals[:r_cmp])
cos = np.abs(np.sum(V_eig[:, :r_cmp] * V_svd[:, :r_cmp], axis=0))   # |cos| removes sign ambiguity
print(f"Max relative eigenvalue difference: {eig_err:.2e}")
print(f"Min |cosine| between matching directions (top 50): {cos[:50].min():.6f}")
summary.update(eig_vs_svd_max_rel_eigenvalue_diff=float(eig_err),
               eig_vs_svd_min_abs_cosine_top50=float(cos[:50].min()),
               rank=int(r))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].semilogy(evals[:r_cmp], "o", ms=3, label="Eigendecomposition")
ax[0].semilogy(evals_svd[:r_cmp], "x", ms=3, label="SVD (S^2/(m-1))")
ax[0].set_xlabel("component"); ax[0].set_ylabel("eigenvalue"); ax[0].legend(); ax[0].set_title("Eigenvalues agree")
ax[1].plot(cos, ".-"); ax[1].set_ylim(0, 1.05); ax[1].set_xlabel("component")
ax[1].set_ylabel("|cos| between directions"); ax[1].set_title("Principal directions agree (up to sign)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/02_eig_vs_svd.png", dpi=130); plt.close()

# Use SVD basis from here on
eigenvalues, V = evals_svd[:r], V_svd[:, :r]

Max relative eigenvalue difference: 1.08e-14
Min |cosine| between matching directions (top 50): 1.000000


### Explained variance and 90% threshold

In [7]:
evr = eigenvalues / eigenvalues.sum()
cum = np.cumsum(evr)
k90 = int(np.searchsorted(cum, 0.90) + 1)
k95 = int(np.searchsorted(cum, 0.95) + 1)
print(f"Components for >=90% variance: {k90}  (cum={cum[k90-1]:.4f});  for 95%: {k95}")
summary.update(k90=k90, k95=k95, variance_at_k90=float(cum[k90 - 1]))

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, r + 1), cum, lw=2)
plt.axhline(0.90, color="r", ls="--", label="90%")
plt.axvline(k90, color="g", ls="--", label=f"k={k90}")
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.title("Cumulative explained variance"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_cumulative_variance.png", dpi=130); plt.close()

Components for >=90% variance: 99  (cum=0.9001);  for 95%: 164


### Eigenfaces

In [8]:
fig, ax = plt.subplots(3, 5, figsize=(10, 7))
for i, a in enumerate(ax.ravel()):
    a.imshow(V[:, i].reshape(H, W), cmap="gray"); a.axis("off")
    a.set_title(f"PC{i+1} ({evr[i]*100:.1f}%)", fontsize=8)
plt.suptitle("Top 15 eigenfaces"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_eigenfaces.png", dpi=130); plt.close()

### Projection, reconstruction and storage

In [9]:
def project(Xin, k):
    return (Xin - mean_face) @ V[:, :k]

def reconstruct(Z):
    return Z @ V[:, :Z.shape[1]].T + mean_face

Ztr, Zte = project(Xtr, k90), project(Xte, k90)
np.savez_compressed(f"{RES_DIR}/pca_reduced_k{k90}.npz", Ztr=Ztr, Zte=Zte, ytr=ytr, yte=yte,
                    mean_face=mean_face, components=V[:, :k90])
print("Reduced representation:", Ztr.shape, Zte.shape)

ks = [k for k in [5, 10, 25, k90, 100, 200] if k <= r]
ks = sorted(set(ks))
rows = []
for k in ks:
    rec = reconstruct(project(Xte, k))
    mse = np.mean((rec - Xte) ** 2)
    psnr = 10 * np.log10(1.0 / mse)
    rows.append(dict(k=k, MSE=mse, PSNR_dB=psnr, variance=cum[k - 1]))
# storage: original = D floats per image; PCA = k floats per image (+ shared model: mean + k components)
n_img = len(Xtr)
orig_floats = n_img * D
for row in rows:
    k = row["k"]
    row["per_image_floats"] = k
    row["compression_ratio_per_image"] = D / k
    row["total_floats_with_model"] = n_img * k + D + k * D
    row["total_vs_original"] = orig_floats / (n_img * k + D + k * D)
import csv
with open(f"{RES_DIR}/reconstruction_storage.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print(f"{'k':>4} {'var':>7} {'MSE':>10} {'PSNR':>7} {'D/k':>8} {'total ratio':>12}")
for r_ in rows:
    print(f"{r_['k']:>4} {r_['variance']:7.3f} {r_['MSE']:10.5f} {r_['PSNR_dB']:7.2f} "
          f"{r_['compression_ratio_per_image']:8.1f} {r_['total_vs_original']:12.2f}")
summary["reconstruction"] = rows

sample = [0, 3, 7, 11]
fig, ax = plt.subplots(len(sample), len(ks) + 1, figsize=(1.8 * (len(ks) + 1), 2.2 * len(sample)))
for ri, si in enumerate(sample):
    ax[ri, 0].imshow(Xte[si].reshape(H, W), cmap="gray"); ax[ri, 0].set_title("original" if ri == 0 else "", fontsize=8)
    for ci, k in enumerate(ks, start=1):
        ax[ri, ci].imshow(reconstruct(project(Xte[si:si+1], k)).reshape(H, W), cmap="gray", vmin=0, vmax=1)
        if ri == 0: ax[ri, ci].set_title(f"k={k}", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.suptitle("Reconstruction of test faces with k components"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_reconstructions.png", dpi=130); plt.close()

plt.figure(figsize=(5.5, 3.8)); plt.plot([r_["k"] for r_ in rows], [r_["PSNR_dB"] for r_ in rows], "o-")
plt.xlabel("k"); plt.ylabel("PSNR (dB)"); plt.title("Reconstruction quality vs k"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/06_psnr_vs_k.png", dpi=130); plt.close()

Reduced representation: (320, 99) (80, 99)
   k     var        MSE    PSNR      D/k  total ratio
   5   0.481    0.01296   18.88   2060.8        51.99
  10   0.601    0.01025   19.89   1030.4        28.29
  25   0.734    0.00756   21.22    412.2        11.95
  99   0.900    0.00497   23.04    104.1         3.10
 100   0.901    0.00496   23.05    103.0         3.07
 200   0.968    0.00421   23.76     51.5         1.54


## Part C: Recognition in PCA space

In [10]:
def sqdist(A, B):
    return (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T

def nn_predict(Atr, ytr_, Ate):
    d = sqdist(Ate, Atr)
    j = d.argmin(1)
    return ytr_[j], j, np.sqrt(np.maximum(d[np.arange(len(Ate)), j], 0))

pred_raw, j_raw, _ = nn_predict(Xtr, ytr, Xte)
acc_raw = float(np.mean(pred_raw == yte))
pred_pca, j_pca, dist_pca = nn_predict(Ztr, ytr, Zte)
acc_pca = float(np.mean(pred_pca == yte))
print(f"1-NN raw pixels ({D} dims):  accuracy = {acc_raw:.4f}")
print(f"1-NN PCA (k={k90} dims):     accuracy = {acc_pca:.4f}")
summary.update(acc_nn_raw=acc_raw, acc_nn_pca_k90=acc_pca)

# Accuracy as a function of k
k_grid = [k for k in [5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 250] if k <= r]
acc_k = []
for k in k_grid:
    p, _, _ = nn_predict(project(Xtr, k), ytr, project(Xte, k))
    acc_k.append(float(np.mean(p == yte)))
summary["nn_accuracy_vs_k"] = dict(zip(map(str, k_grid), acc_k))
plt.figure(figsize=(6, 4)); plt.plot(k_grid, acc_k, "o-", label="1-NN in PCA space")
plt.axhline(acc_raw, color="r", ls="--", label="1-NN on raw pixels")
plt.axvline(k90, color="g", ls=":", label=f"k90={k90}")
plt.xlabel("k"); plt.ylabel("Test accuracy"); plt.legend(); plt.grid(alpha=.3)
plt.title("Recognition accuracy vs number of components"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/07_nn_accuracy_vs_k.png", dpi=130); plt.close()

# Correct / incorrect match examples
correct = np.where(pred_pca == yte)[0]
wrong = np.where(pred_pca != yte)[0]
show = list(correct[:4]) + list(wrong[:4])
if show:
    fig, ax = plt.subplots(2, len(show), figsize=(2 * len(show), 4.6), squeeze=False)
    for c, i in enumerate(show):
        ax[0, c].imshow(Xte[i].reshape(H, W), cmap="gray"); ax[0, c].axis("off")
        ax[0, c].set_title(f"test s{yte[i]}", fontsize=8)
        ax[1, c].imshow(Xtr[j_pca[i]].reshape(H, W), cmap="gray"); ax[1, c].axis("off")
        ok = pred_pca[i] == yte[i]
        ax[1, c].set_title(f"match s{pred_pca[i]}\n{'CORRECT' if ok else 'WRONG'}", fontsize=8,
                           color="green" if ok else "red")
    plt.suptitle("Top row: query | bottom row: closest training face (PCA space)")
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/08_matches.png", dpi=130); plt.close()
print("Misclassified test images:", [(int(yte[i]), int(pred_pca[i])) for i in wrong])
summary["pca_nn_errors"] = [[int(yte[i]), int(pred_pca[i])] for i in wrong]

1-NN raw pixels (10304 dims):  accuracy = 0.9875
1-NN PCA (k=99 dims):     accuracy = 0.9875
Misclassified test images: [(32, 2)]


## Part D: Traditional ML 
Hyper-parameters and k are chosen by 4-fold stratified CV **on the training set only**;
the test set is used once for reporting.

In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# Fold count must not exceed images per class in the training set
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
K_CANDS = [k for k in [10, 20, 30, k90, 50, 75, 100] if k <= r]
K_CANDS = sorted(set(K_CANDS))

def make_models():
    return {
        "KNN": (KNeighborsClassifier(), {"n_neighbors": [1, 3, 5], "weights": ["uniform", "distance"]}),
        "Logistic Regression": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                                {"logisticregression__C": [0.01, 0.1, 1, 10]}),
        "SVM (linear)": (make_pipeline(StandardScaler(), SVC(kernel="linear")),
                         {"svc__C": [0.001, 0.01, 0.1, 1]}),
        "SVM (RBF)": (SVC(kernel="rbf"), {"C": [1, 10, 100], "gamma": ["scale", 0.1, 0.01]}),
    }

# Baseline defaults at k90
table = [dict(model="Nearest neighbour (NumPy, Euclidean)", features=f"raw pixels ({D})", k=D,
              params="-", cv_acc=np.nan, test_acc=acc_raw),
         dict(model="Nearest neighbour (NumPy, Euclidean)", features=f"PCA k={k90}", k=k90,
              params="-", cv_acc=np.nan, test_acc=acc_pca)]

best_overall = None
sweep = {}   # model -> {k: test acc of CV-best params}
for name, (est, grid) in make_models().items():
    sweep[name] = {}
    best_for_model = None
    for k in K_CANDS:
        A, B = project(Xtr, k), project(Xte, k)
        gs = GridSearchCV(est, grid, cv=cv, n_jobs=-1).fit(A, ytr)
        te = float(gs.score(B, yte))
        sweep[name][k] = te
        rec = dict(model=name, features=f"PCA k={k}", k=k, params=str(gs.best_params_),
                   cv_acc=float(gs.best_score_), test_acc=te)
        if best_for_model is None or rec["cv_acc"] > best_for_model["cv_acc"]:
            best_for_model = rec
    table.append(best_for_model)
    print(f"{name:22s} best k={best_for_model['k']:>3}  CV={best_for_model['cv_acc']:.4f}  "
          f"test={best_for_model['test_acc']:.4f}  {best_for_model['params']}")

with open(f"{RES_DIR}/ml_comparison.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(table[0].keys())); w.writeheader(); w.writerows(table)
summary["ml_comparison"] = table

md = ["| Model | Features | k | Best params (CV) | CV acc | Test acc |", "|---|---|---|---|---|---|"]
for t in table:
    cvs = "-" if np.isnan(t["cv_acc"]) else f"{t['cv_acc']:.3f}"
    md.append(f"| {t['model']} | {t['features']} | {t['k']} | {t['params']} | {cvs} | {t['test_acc']:.3f} |")
open(f"{RES_DIR}/ml_comparison.md", "w").write("\n".join(md) + "\n")
print("\n".join(md))

plt.figure(figsize=(7, 4.2))
for name, d in sweep.items():
    plt.plot(list(d.keys()), list(d.values()), "o-", label=name)
plt.plot(k_grid, acc_k, "k--", label="1-NN (NumPy)")
plt.xlabel("PCA components k"); plt.ylabel("Test accuracy"); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.title("Classifier accuracy vs retained components"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/09_classifier_comparison.png", dpi=130); plt.close()

with open(f"{RES_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=lambda o: None if (isinstance(o, float) and np.isnan(o)) else float(o))
print("Done. Figures in ./figures, tables in ./results")

d:\annaconda\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "d:\annaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "d:\annaconda\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\annaconda\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "d:\annaconda\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^

KNN                    best k= 30  CV=0.9688  test=0.9875  {'n_neighbors': 1, 'weights': 'uniform'}
Logistic Regression    best k= 50  CV=0.9594  test=0.9500  {'logisticregression__C': 0.01}
SVM (linear)           best k= 30  CV=0.9594  test=0.9625  {'svc__C': 0.1}
SVM (RBF)              best k= 75  CV=0.9813  test=0.9875  {'C': 10, 'gamma': 'scale'}
| Model | Features | k | Best params (CV) | CV acc | Test acc |
|---|---|---|---|---|---|
| Nearest neighbour (NumPy, Euclidean) | raw pixels (10304) | 10304 | - | - | 0.988 |
| Nearest neighbour (NumPy, Euclidean) | PCA k=99 | 99 | - | - | 0.988 |
| KNN | PCA k=30 | 30 | {'n_neighbors': 1, 'weights': 'uniform'} | 0.969 | 0.988 |
| Logistic Regression | PCA k=50 | 50 | {'logisticregression__C': 0.01} | 0.959 | 0.950 |
| SVM (linear) | PCA k=30 | 30 | {'svc__C': 0.1} | 0.959 | 0.963 |
| SVM (RBF) | PCA k=75 | 75 | {'C': 10, 'gamma': 'scale'} | 0.981 | 0.988 |
Done. Figures in ./figures, tables in ./results
